In [1]:
"""
Bayesian Autoregressive (BAR) Forecasting Model
=================================================
Real-time recursive forecasts of log real TTF NG prices.
Three specifications: BAR(1), BAR(12), BAR(AIC, p≤6)

Minnesota prior (GLP 2015): φ_i ~ N(0, (λ/i)² · σ²), c ~ N(0, 10⁶)
λ selected by maximising log marginal likelihood at each origin.
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from statsmodels.tsa.ar_model import AutoReg
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START  = "2015-01-01"
AIC_P_MAX   = 6
INPUT_FILE  = "Input_TTF_NG_Real_Average_Prices.xlsx"
OUTPUT_FILE = "Output_BAR_forecasts.xlsx"

SPECIFICATIONS = [
    ("BAR(1)",        1),
    ("BAR(12)",       12),
    ("BAR(AIC,p≤6)", None),
]

In [3]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])
df = df[["date","price_real"]].sort_values("date").reset_index(drop=True)
df["log_price"] = np.log(df["price_real"])

def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["price_real"].values[0] if len(m) == 1 else np.nan

In [4]:
# ── Core functions ────────────────────────────────────────────────────────────
def build_X_y(y, p):
    """Build regressor matrix X (with intercept) and response vector."""
    T = len(y)
    X = np.ones((T - p, p + 1))
    for i in range(p):
        X[:, i + 1] = y[p - i - 1: T - i - 1]
    return X, y[p:]

def log_marginal_likelihood(lambda_val, y, p, sigma2):

# Log marginal likelihood for Minnesota prior — used to select λ.
    if lambda_val <= 1e-6:
        return -np.inf
    X, y_dep = build_X_y(y, p)

    # Prior precision matrix V0_inv
    V0_diag = np.array(
        [1e6] + [(lambda_val / i) ** 2 * sigma2 for i in range(1, p + 1)]
    )
    V0_inv = np.diag(1.0 / V0_diag)

    # Posterior precision and mean
    V1_inv = V0_inv + X.T @ X / sigma2
    V1     = np.linalg.inv(V1_inv)
    m1     = V1 @ (X.T @ y_dep / sigma2)

    # Log marginal likelihood (up to constants)
    _, logdet_V0 = np.linalg.slogdet(np.diag(V0_diag))
    _, logdet_V1 = np.linalg.slogdet(V1)
    lml = (0.5 * logdet_V1
           - 0.5 * logdet_V0
           - 0.5 / sigma2 * (y_dep @ y_dep - m1 @ V1_inv @ m1))
    return lml

def bar_estimate(y, p):

    # Prior scale: variance of first differences (standard Minnesota choice)
    sigma2 = max(np.var(np.diff(y)), 1e-10)

    # Optimise lambda by maximising log marginal likelihood
    result = minimize_scalar(
        lambda lam: -log_marginal_likelihood(lam, y, p, sigma2),
        bounds=(0.001, 100),
        method="bounded"
    )
    lambda_opt = result.x

    # Posterior mean
    X, y_dep = build_X_y(y, p)
    V0_diag  = np.array(
        [1e6] + [(lambda_opt / i) ** 2 * sigma2 for i in range(1, p + 1)]
    )
    V0_inv = np.diag(1.0 / V0_diag)
    V1_inv = V0_inv + X.T @ X / sigma2
    V1     = np.linalg.inv(V1_inv)
    m1     = V1 @ (X.T @ y_dep / sigma2)

    return m1, lambda_opt   # m1 = [intercept, phi_1, ..., phi_p]

def select_aic_lag(y, p_max):
    """Select lag order by AIC (same as AR model)."""
    best_aic, best_p = np.inf, 1
    for p in range(1, p_max + 1):
        try:
            m = AutoReg(y, lags=p, old_names=False).fit()
            if m.aic < best_aic:
                best_aic, best_p = m.aic, p
        except Exception:
            pass
    return best_p

def iterated_forecast(params, history, p, horizons):
    """
    Iterated multi-step forecast using posterior mean coefficients.
    Identical logic to AR — only the coefficients differ.
    """
    c    = params[0]
    phis = params[1:]
    buf  = list(history[-p:])
    h_max = max(horizons)
    fcsts = {}
    for h in range(1, h_max + 1):
        yhat = c + np.dot(phis, buf[-p:][::-1])
        buf.append(yhat)
        if h in horizons:
            fcsts[h] = yhat
    return fcsts

In [5]:
# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []
origins = df[df["date"] >= EVAL_START]["date"].tolist()

for origin_date in origins:
    history = df[df["date"] <= origin_date]["log_price"].values

    for label, fixed_p in SPECIFICATIONS:

        # Determine lag order
        if fixed_p is not None:
            p = fixed_p
        else:
            p = select_aic_lag(history, AIC_P_MAX)

        if len(history) <= p + 1:
            continue

        try:
            params, lambda_opt = bar_estimate(history, p)
            forecasts = iterated_forecast(params, history, p, HORIZONS)
        except Exception:
            continue

        for h in HORIZONS:
            actual_ym      = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            forecast_level = np.exp(forecasts[h])
            actual_val     = get_actual(actual_ym)

            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        forecast_level,
                "actual":          actual_val,
                "lag_order_used":  p,
                "lambda_opt":      round(lambda_opt, 4),
            })

In [6]:
# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

In [7]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("BAR forecasting complete.")
print(f"  Specifications:   {[s[0] for s in SPECIFICATIONS]}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Horizons:         {HORIZONS}")
print(f"  Total rows:       {len(results)}")
print(f"  Output:           {OUTPUT_FILE}")
print()

BAR forecasting complete.
  Specifications:   ['BAR(1)', 'BAR(12)', 'BAR(AIC,p≤6)']
  Forecast origins: 132
  Horizons:         [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Total rows:       3564
  Output:           Output_BAR_forecasts.xlsx



In [8]:
# Sample — first origin
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used","lambda_opt",
              "actual_month","forecast","actual"]].to_string(index=False))

Sample — first origin (2015-01-31):
       model  horizon  lag_order_used  lambda_opt actual_month  forecast    actual
      BAR(1)        1               1     60.6311      2015-02 19.808977 22.938516
      BAR(1)        3               1     60.6311      2015-04 19.877914 22.046423
      BAR(1)        6               1     60.6311      2015-07 19.960968 20.679393
      BAR(1)        9               1     60.6311      2015-10 20.024629 18.160884
      BAR(1)       12               1     60.6311      2016-01 20.073382 13.882111
      BAR(1)       15               1     60.6311      2016-04 20.110694 12.101522
      BAR(1)       18               1     60.6311      2016-07 20.139235 14.115551
      BAR(1)       21               1     60.6311      2016-10 20.161059 15.958961
      BAR(1)       24               1     60.6311      2017-01 20.177741 19.873384
     BAR(12)        1              12     89.0260      2015-02 19.644891 22.938516
     BAR(12)        3              12     89.0260  

In [9]:
# Lambda distribution
print("\nOptimal lambda summary by model:")
for label, _ in SPECIFICATIONS:
    sub = results[results["model"] == label].drop_duplicates("forecast_origin")
    print(f"  {label}: mean={sub['lambda_opt'].mean():.2f}  "
          f"min={sub['lambda_opt'].min():.2f}  "
          f"max={sub['lambda_opt'].max():.2f}")


Optimal lambda summary by model:
  BAR(1): mean=54.22  min=32.87  max=70.56
  BAR(12): mean=60.48  min=41.90  max=92.73
  BAR(AIC,p≤6): mean=49.30  min=23.24  max=74.32


In [10]:
# BAR(AIC) lag distribution
aic_df = results[results["model"] == "BAR(AIC,p≤6)"].drop_duplicates("forecast_origin")
if not aic_df.empty:
    print("\nBAR(AIC) lag order distribution:")
    print(aic_df["lag_order_used"].value_counts().sort_index().to_string())


BAR(AIC) lag order distribution:
lag_order_used
1    15
2    25
3    16
4     3
5    73
